# All SQL queries run in BigQuery

1. Create Embedding Model in BigQuery
```
CREATE OR REPLACE MODEL `aurora_bay.Embeddings`
REMOTE WITH CONNECTION `us.embedding_conn`
OPTIONS (ENDPOINT = 'text-embedding-005');
```
2. Load Aurora Bay FAQ Data into BigQuery
```
LOAD DATA OVERWRITE `aurora_bay.faqs`
(
  question STRING,
  answer STRING
)
FROM FILES (
  format = 'CSV',
  uris = ['gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv'],
  skip_leading_rows = 1
);
```
3. Generate and store FAQ Embeddings
```
CREATE OR REPLACE TABLE `aurora_bay.faqs_embedded` AS
SELECT *
FROM ML.GENERATE_EMBEDDING(
  MODEL `aurora_bay.Embeddings`,
  (
    SELECT
      CONCAT(question, ' ', answer) AS content
    FROM `aurora_bay.faqs`
  )
);
```
4. Vector search
```
SELECT
  base.content,
  distance
FROM VECTOR_SEARCH(
  TABLE `aurora_bay.faqs_embedded`,
  'ml_generate_embedding_result',
  (
    SELECT
      ml_generate_embedding_result,
      'Where is the Aurora Bay Town Hall located?' AS content
    FROM ML.GENERATE_EMBEDDING(
      MODEL `aurora_bay.Embeddings`,
      (SELECT 'Where is the Aurora Bay Town Hall located?' AS content)
    )
  ),
  top_k => 5
);
```

In [6]:
from google.cloud import bigquery
import vertexai
from vertexai.generative_models import GenerativeModel

# Initialize BigQuery client
bq_client = bigquery.Client()

# Initialize Vertex AI
PROJECT_ID = "qwiklabs-gcp-02-5d502ebd5e20"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)

# Load Gemini model
model = GenerativeModel("gemini-2.5-pro")


In [13]:
load_faqs_sql = """
CREATE OR REPLACE TABLE `aurora_bay.faqs` (
  question STRING,
  answer STRING
);

LOAD DATA OVERWRITE `aurora_bay.faqs`
(
  question STRING,
  answer STRING
)
FROM FILES (
  format = 'CSV',
  uris = ['gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv'],
  skip_leading_rows = 1
);
"""

job = bq_client.query(load_faqs_sql)
job.result()

print("Aurora Bay FAQ data loaded successfully.")


Aurora Bay FAQ data loaded successfully.


In [14]:
create_model_sql = """
CREATE OR REPLACE MODEL `aurora_bay.Embeddings`
REMOTE WITH CONNECTION `us.embedding_conn`
OPTIONS (
  endpoint = 'text-embedding-005'
);
"""

job = bq_client.query(create_model_sql)
job.result()

print("Embedding model created successfully.")


Embedding model created successfully.


In [15]:
generate_embeddings_sql = """
CREATE OR REPLACE TABLE `aurora_bay.faqs_embedded` AS
SELECT *
FROM ML.GENERATE_EMBEDDING(
  MODEL `aurora_bay.Embeddings`,
  (
    SELECT
      CONCAT(question, ' ', answer) AS content
    FROM `aurora_bay.faqs`
  )
);
"""

job = bq_client.query(generate_embeddings_sql)
job.result()

print("FAQ embeddings generated and stored successfully.")

FAQ embeddings generated and stored successfully.


In [16]:
verify_sql = """
SELECT COUNT(*) AS row_count
FROM `aurora_bay.faqs_embedded`
"""

df = bq_client.query(verify_sql).to_dataframe()
df

,row_count
0,50


In [17]:
def retrieve_faq_context(user_question, top_k=5):

    query = f"""
    SELECT
      base.content
    FROM VECTOR_SEARCH(
      TABLE `aurora_bay.faqs_embedded`,
      'ml_generate_embedding_result',
      (
        SELECT
          ml_generate_embedding_result,
          '{user_question}' AS content
        FROM ML.GENERATE_EMBEDDING(
          MODEL `aurora_bay.Embeddings`,
          (SELECT '{user_question}' AS content)
        )
      ),
      top_k => {top_k}
    );
    """

    results = bq_client.query(query).result()

    retrieved_chunks = []
    for row in results:
        retrieved_chunks.append(row.content)

    return "\n\n".join(retrieved_chunks)

In [18]:
def generate_answer(user_question, context):

    prompt = f"""
You are a chatbot for the town of Aurora Bay, Alaska.

STRICT RULES:
- Answer the question using ONLY the context provided.
- Do NOT use outside knowledge.
- If the answer is not in the context, respond: "I do not have that information."

Context:
{context}

Question:
{user_question}

Answer:
"""

    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": 0.2,
            "max_output_tokens": 512
        }
    )

    return response.text.strip()

In [19]:
def aurora_bay_chatbot(user_question):
    """
    End-to-end RAG chatbot:
    1. Retrieve relevant FAQ context from BigQuery
    2. Generate grounded answer using Gemini
    """

    context = retrieve_faq_context(user_question)

    if not context.strip():
        return "I do not have that information."

    answer = generate_answer(user_question, context)
    return answer

In [21]:
# Example chatbot tests

questions = [
    "Where is the Aurora Bay Town Hall located?",
    "How can I contact Aurora Bay city offices?",
    "What is the weather today?"
]

for q in questions:
    print(f"Question: {q}")
    print(f"Answer: {aurora_bay_chatbot(q)}")
    print("-" * 50)


Question: Where is the Aurora Bay Town Hall located?
Answer: The Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.
--------------------------------------------------
Question: How can I contact Aurora Bay city offices?
Answer: The Town Hall is located at 100 Harbor View Road. The Aurora Bay Fire
--------------------------------------------------
Question: What is the weather today?
Answer: I do not have that information.
--------------------------------------------------
